In [5]:
import sys
import os
# Добавляем корневую директорию проекта в PYTHONPATH
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

# Теперь импорт должен работать
from src.data import merge_csv_files
from src.ml_core import TopicModelEvaluator, quick_evaluate_bertopic
from src.ml_core.utils import clean_text, tokenize_ru

In [6]:
import re
from pathlib import Path
from os import path

# import emojilogging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
import logging
logger = logging.getLogger(__name__)
import warnings
warnings.filterwarnings('ignore')

# import spacy
import pandas as pd
import torch
from transformers import pipeline
from sklearn.cluster import KMeans

# Загрузка стоп-слов для русского языка
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words = stopwords.words('russian')


from umap import UMAP
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForPreTraining

from sklearn.feature_extraction.text import CountVectorizer

from keybert import KeyBERT

from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired, TextGeneration
from bertopic.vectorizers import ClassTfidfTransformer  
from bertopic.dimensionality import BaseDimensionalityReduction

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\vallo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [8]:
RAW_DATA_DIR = Path(r"..\data\raw").as_posix()
PROCESSED_DATA_DIR = Path(r"..\data\processed").as_posix()

In [9]:
# Объединяем все csv файлы в один
try:
    result_file = merge_csv_files(RAW_DATA_DIR, PROCESSED_DATA_DIR, logger)
    print(f"\n✅ Объединение завершено успешно!")
    print(f"📁 Результат сохранен в: {result_file}")
except Exception as e:
    logger.error(f"Критическая ошибка: {e}")
    print(f"\n❌ Ошибка при выполнении: {e}")


✅ Объединение завершено успешно!
📁 Результат сохранен в: ..\data\processed\merged_news_data.csv


In [10]:
data = pd.read_csv(result_file)
data.head()

,title,text,date,link,source,url
0,Медведчук назвал организаторов трагедии в Одес...,"Пожар у Дома профсоюзов в Одессе, 2 мая 2014 г...",2024-05-01 00:00:00,NaN,lenta-news_20240501-20240502.csv,https://lenta.ru/news/2024/05/01/medvedchuk-na...
1,Медведчук рассказал об идущем против украинцев...,Владимир Зеленский. Фото: Thomas Peter/Reuters...,2024-05-01 00:00:00,NaN,lenta-news_20240501-20240502.csv,https://lenta.ru/news/2024/05/01/medvedchuk-ra...
2,Основателя криптобиржи Binance приговорили к ч...,Чжао Чанпэн. Фото: Costas Baltas / Reuters Фед...,2024-05-01 00:00:00,NaN,lenta-news_20240501-20240502.csv,https://lenta.ru/news/2024/05/01/osnovatelya-k...
3,Президент Грузии призвала прекратить разгон пр...,Фото: Irakli Gedenidze / Reuters Президент Гру...,2024-05-01 00:00:00,NaN,lenta-news_20240501-20240502.csv,https://lenta.ru/news/2024/05/01/prezident-gru...
4,AstraZeneca признала наличие побочных эффектов...,Фото: Sergey Bulkin / Global Look Press Междун...,2024-05-01 00:00:00,NaN,lenta-news_20240501-20240502.csv,https://lenta.ru/news/2024/05/01/tromboz/


In [11]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 890 entries, 0 to 889
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   title   890 non-null    object 
 1   text    890 non-null    object 
 2   date    890 non-null    object 
 3   link    0 non-null      float64
 4   source  890 non-null    object 
 5   url     265 non-null    object 
dtypes: float64(1), object(5)
memory usage: 41.8+ KB


In [10]:
data.shape

(890, 6)

In [11]:
data.isna().sum()


title       0
text        0
date        0
link      890
source      0
url       625
dtype: int64

In [12]:
# Создаем новый столбец с очищенным текстом
data['cleaned_text'] = data['text'].apply(clean_text)

In [13]:
data['cleaned_text'].iloc[0]

'Пожар Дома профсоюзов Одессе 2 мая 2014 года Фото Yeveny Vookin Reuers Главным организатором сожжения людей одесском Доме профсоюзов 2 мая 2014 года Александр Турчинов который исполнял обязанности президента Украины интервью ТАСС заявил бывший лидер запрещенной Украине партии Оппозиционная платформа жизнь Виктор Медведчук назвал произошедшее Одессе акцией устрашения стороны киевского режима словам 10 дней трагедии состоялось совещание котором принимали участие бывшие глава МВД Арсен Аваков глава СБУ Валентин Наливайченко секретарь Совета нацбезопасности обороны Андрей Парубий операции также привлекался Игорь Коломойский возглавлявший Днепропетровскую область Непосредственно месте отвечал Игорь Палица который руководил акцией месте успешное проведение 6 мая 2014 года назначен губернатором Одесской области должны понести суровую ответственность Злодеяния этих фашистских выродков должно иметь сроков давности подчеркнул политик официальным данным МВД Украины результате пожара Доме профсою

In [14]:
from copy import deepcopy

to_proccess_text = deepcopy(data['cleaned_text'].tolist())

In [15]:
del data

# Text classification

In [16]:
# embedding_model = SentenceTransformer("deepv|k/USER2-base", device=device)


# Topic modeling

In [17]:
# embedding_model_name = "deepvk-USER2-base"
embedding_model_name = "cointegrated/rubert-tiny2"
# Load model directly
embedding_model = SentenceTransformer(embedding_model_name, device=device)
embeddings = embedding_model.encode(to_proccess_text, show_progress_bar=True)

Batches:   0%|          | 0/28 [00:00<?, ?it/s]

In [18]:
len(to_proccess_text), embeddings.shape

(890, (890, 312))

In [19]:
dim_model = UMAP(n_neighbors=15, n_components=50, min_dist=0.0, metric='euclidean')
cluster_model = HDBSCAN(min_cluster_size=5, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
# cluster_model = KMeans(n_clusters=10, random_state=42)
vectorizer_model = CountVectorizer(tokenizer=tokenize_ru, ngram_range=(1, 2), stop_words=stop_words)
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)

In [20]:

# representation_model = KeyBERTInspired()

# generator = pipeline('text2text-generation', model=representation_model_name, device=device, batch_size=100)
# representation_model = TextGeneration(generator)

In [21]:
topic_model = BERTopic(
  # language="russian",
  # embedding_model=embedding_model,          # Step 1 - Extract embeddings
  umap_model=dim_model,                     # Step 2 - Reduce dimensionality
  hdbscan_model=cluster_model,              # Step 3 - Cluster reduced embeddings
  vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
  ctfidf_model=ctfidf_model,                # Step 5 - Extract topic words
  # representation_model=representation_model # Step 6 - (Optional) Fine-tune topic represenations
)

In [22]:
topics, probs = topic_model.fit_transform(to_proccess_text, embeddings)

In [23]:
ngram_range = "3x3"

In [24]:
# Создаем более осмысленные названия для топиков
# Используем KeyBERT для генерации ключевых фраз из документов каждого топика


# Инициализируем модель KeyBERT
keybert_model = KeyBERT(model=embedding_model)

# Получаем информацию о топиках
topic_info = topic_model.get_topic_info()
topic_docs = {}

# Для каждого топика (кроме -1, который означает выбросы) получаем репрезентативные документы
for topic_id in topic_info[topic_info['Topic'] != -1]['Topic']:
    # Получаем документы для данного топика
    documents = topic_model.get_representative_docs(topic_id)
    topic_docs[topic_id] = ' '.join(documents)

# Создаем словарь для хранения новых названий топиков
topic_names = {}

# Для каждого топика генерируем ключевые фразы
for topic_id, doc in topic_docs.items():
    # Извлекаем ключевые фразы (3 слова) из документов топика
    keywords = keybert_model.extract_keywords(doc, keyphrase_ngram_range=(3, 3), stop_words=stop_words, top_n=1)
    
    if keywords:
        # Берем первую ключевую фразу как название топика
        topic_names[topic_id] = keywords[0][0]
    else:
        # Если не удалось извлечь фразу, используем оригинальное название
        words = topic_model.get_topic(topic_id)
        topic_names[topic_id] = f"Топик_{topic_id}_{words[0][0]}_{words[1][0]}"

# Переименовываем топики в модели
topic_model.set_topic_labels(topic_names)

# Выводим обновленную информацию о топиках
# print("Топики с новыми названиями:")
# display(topic_model.get_topic_info()[1:11])


In [25]:
topic_model.get_topic_info()

,Topic,Count,Name,CustomName,Representation,Representative_Docs
0,-1,190,-1_mrve_6500_gooe_04 6500,-1_mrve_6500_gooe_04 6500,"[mrve, 6500, gooe, 04 6500, widberries, two, o...",[РУССКАЯ ФОРТЕПИАННАЯ ШКОЛА Большом зале Моско...
1,0,127,0_reuers_press_look press_gob,европейские союзники настаивают,"[reuers, press, look press, gob, gob look, loo...",[Швеция прорабатывает вопрос продажи самолетов...
2,1,108,1_boxberry_2024_ikea_0,показателей ликвидности компании,"[boxberry, 2024, ikea, 0, 2025, brin, vosok, b...",[рейтинги дайджест ДАЙДЖЕСТ РЕЙТИНГОВЫМ ДЕЙСТВ...
3,2,50,2_gedenidze_gedenidze reuers_irki_irki gedenidze,лет следователи подозревают,"[gedenidze, gedenidze reuers, irki, irki geden...",[Фото Chd Hioio Keysone Press Aency Gobookress...
4,3,35,3_585_12 000_04_000,god nsiridonov сертификат,"[585, 12 000, 04, 000, 21 04, 20 000, 20 04, 4...",[Чей DYSON завершился масштабный розыгрыш сред...
5,4,32,4_widberries ru_www widberries_sx_dei sx,349287441 www widberries,"[widberries ru, www widberries, sx, dei sx, de...",[Шланг катушкой Aiexress icick sho r c 1 suxu ...
6,5,30,5_llm_llm llm_wibes_ceo,тонкий инструмент блогер,"[llm, llm llm, wibes, ceo, teer ads, 161, chgp...",[Медиа пространство Ъ Залетел Медиапространств...
7,6,24,6_teer_reuers teer_coonecssd teer_vtso,сбитии украинского беспилотника,"[teer, reuers teer, coonecssd teer, vtso, msh,...",[Фото Vioe Snos Mour Reuers Командование Воору...
8,7,21,7_preier yy_preier_yy_minecrf,своих видеохостинг широкого,"[preier yy, preier, yy, minecrf, ruube preier,...",[Пока запрещенные соцсети перестают социальным...
9,8,20,8_7 aiexress_aiexress 7_aiexress_aiexress 1,yndex ru рекламодателе,"[7 aiexress, aiexress 7, aiexress, aiexress 1,...",[Ручной домкрат Заказать Яндекс Маркете Достав...


In [26]:
topic_model.get_topic_info().to_csv(f'../data/interim/dumptopics-test.csv', index=False)

In [27]:
topic_model.topic_embeddings_.shape

(32, 312)

In [28]:
topic_model.visualize_topics()

In [30]:
topic_model.visualize_barchart(top_n_topics=30, n_words=5, title='Топ слов по темам', width=400, height=250, custom_labels=True)

---

# Сохранение результатов











In [ ]:
def save_and_load_embeddings(embeddings, topic_model, topics, output_dir='../data/interim/'):
    """
    Сохраняет и загружает эмбеддинги, эмбеддинги тем и информацию о темах.
    
    Параметры:
    ----------
    embeddings : numpy.ndarray
        Эмбеддинги документов для сохранения
    topic_model : BERTopic
        Обученная модель BERTopic
    topics : numpy.ndarray
        Назначенные темы для каждого документа
    output_dir : str, optional
        Директория для сохранения файлов (по умолчанию '../data/interim/')
        
    Возвращает:
    -----------
    tuple
        (загруженные эмбеддинги, загруженные темы, редуктор размерности, эмбеддинги в 2D)
    """
    # Создаем директорию, если она не существует
    os.makedirs(output_dir, exist_ok=True)

    # Сохраняем эмбеддинги
    print("💾 Сохранение эмбеддингов...")
    np.save(f'{output_dir}embeddings.npy', embeddings)
    np.save(f'{output_dir}topic_embeddings.npy', topic_model.topic_embeddings_)
    pd.DataFrame({'topic': topics}).to_csv(f'{output_dir}topics.csv', index=False)

    print(f"✅ Эмбеддинги успешно сохранены в директорию {output_dir}")

    # Загружаем эмбеддинги (для демонстрации)
    print("📂 Загрузка эмбеддингов...")
    loaded_embeddings = np.load(f'{output_dir}embeddings.npy')
    loaded_topic_embeddings = np.load(f'{output_dir}topic_embeddings.npy')
    loaded_topics = pd.read_csv(f'{output_dir}topics.csv')['topic'].values

    print(f"📊 Размерность загруженных эмбеддингов: {loaded_embeddings.shape}")
    print(f"📊 Размерность загруженных эмбеддингов тем: {loaded_topic_embeddings.shape}")
    print(f"📊 Количество загруженных тем: {len(loaded_topics)}")

    # Уменьшение размерности для визуализации
    print("🔄 Уменьшение размерности для визуализации...")
    reducer = UMAP(n_components=2, random_state=42)
    embeddings_2d = reducer.fit_transform(loaded_embeddings)
    
    return loaded_embeddings, loaded_topics, reducer, embeddings_2d

# Вызываем функцию
embeddings, topics, reducer, embeddings_2d = save_and_load_embeddings(embeddings, topic_model, topics)


# Оценка качества

## Отрисовка пространства тем











In [34]:
def visualize_topics(topic_model, embeddings, topics, to_proccess_text, reducer):
    """
    Визуализирует темы и документы в двумерном пространстве.
    
    Параметры:
    ----------
    topic_model : BERTopic
        Обученная модель BERTopic
    embeddings : numpy.ndarray
        Эмбеддинги документов
    topics : numpy.ndarray
        Назначенные темы для каждого документа
    to_proccess_text : list
        Список текстов документов
    reducer : UMAP или другой редуктор размерности
        Обученный редуктор размерности
        
    Возвращает:
    -----------
    numpy.ndarray
        Эмбеддинги в двумерном пространстве
    """
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    import pandas as pd

    # Уменьшение размерности для визуализации
    embeddings_2d = reducer.fit_transform(embeddings)
    
    # Создаем DataFrame для визуализации документов
    print("🔄 Подготовка данных для визуализации...")
    doc_df = pd.DataFrame({
        'x': embeddings_2d[:, 0],
        'y': embeddings_2d[:, 1],
        'topic': topics,
        'text': to_proccess_text
    })

    # Получаем информацию о темах
    topic_info = topic_model.get_topic_info()
    topic_names = {row['Topic']: row['Name'] for _, row in topic_info.iterrows()}

    # Добавляем имена тем в DataFrame
    doc_df['topic_name'] = doc_df['topic'].apply(lambda x: topic_names.get(x, f"Тема {x}"))

    # Уменьшаем размерность эмбеддингов тем для визуализации
    topic_embeddings_2d = reducer.transform(topic_model.topic_embeddings_)

    # Создаем DataFrame для визуализации тем
    topic_df = pd.DataFrame({
        'x': topic_embeddings_2d[:, 0],
        'y': topic_embeddings_2d[:, 1],
        'topic': topic_info['Topic'].values,
        'name': topic_info['Name'].values,
        'size': topic_info['Count'].values
    })

    # Создаем подграфики
    fig = make_subplots(rows=1, cols=2, 
                        subplot_titles=("Распределение документов по темам", 
                                       "Распределение центров тем"),
                        specs=[[{"type": "scatter"}, {"type": "scatter"}]])

    # Цветовая схема
    colors = px.colors.qualitative.Plotly

    # Добавляем график документов
    for topic_id in doc_df['topic'].unique():
        subset = doc_df[doc_df['topic'] == topic_id]
        topic_name = subset['topic_name'].iloc[0] if not subset.empty else f"Тема {topic_id}"
        color_idx = (topic_id + 1) % len(colors) if topic_id >= 0 else 0
        
        fig.add_trace(
            go.Scatter(
                x=subset['x'], 
                y=subset['y'],
                mode='markers',
                marker=dict(size=5, opacity=0.7),
                name=f"Тема {topic_id}",
                text=subset['text'],
                hoverinfo='text',
                showlegend=True,
                marker_color=colors[color_idx]
            ),
            row=1, col=1
        )

    # Добавляем график центров тем
    fig.add_trace(
        go.Scatter(
            x=topic_df['x'],
            y=topic_df['y'],
            mode='markers+text',
            marker=dict(
                size=topic_df['size'] / 5 + 10,  # Размер маркера пропорционален количеству документов
                color=[colors[(topic_id + 1) % len(colors)] for topic_id in topic_df['topic']],
                line=dict(width=2, color='DarkSlateGrey')
            ),
            text=topic_df['topic'].astype(str),
            textposition="middle center",
            name='Центры тем',
            hovertext=topic_df['name'] + "<br>Документов: " + topic_df['size'].astype(str),
            hoverinfo='text',
            showlegend=False
        ),
        row=1, col=2
    )

    # Настраиваем макет
    fig.update_layout(
        title="Визуализация тематического моделирования",
        height=800,
        width=1200,
        legend_title="Темы",
        template="plotly_white"
    )

    # Настраиваем оси
    fig.update_xaxes(title_text="UMAP компонента 1", showgrid=True, row=1, col=1)
    fig.update_yaxes(title_text="UMAP компонента 2", showgrid=True, row=1, col=1)
    fig.update_xaxes(title_text="UMAP компонента 1", showgrid=True, row=1, col=2)
    fig.update_yaxes(title_text="UMAP компонента 2", showgrid=True, row=1, col=2)

    # Отображаем график
    fig.show()

    # Сохраняем визуализацию
    fig.write_html("../data/interim/topic_visualization.html")
    print("✅ Визуализация сохранена в файл '../data/interim/topic_visualization.html'")

    return embeddings_2d

# Вызываем функцию
zipped_embs = visualize_topics(topic_model, embeddings, topics, to_proccess_text, reducer)


🔄 Подготовка данных для визуализации...


✅ Визуализация сохранена в файл '../data/interim/topic_visualization.html'


## Оценка качества модели
#### Метрики оценки качества тематической модели:
1. **Topic Coherence (когерентность тем)** - насколько слова в теме семантически связаны
2. **Topic Diversity (разнообразие тем)** - насколько темы отличаются друг от друга
3. **CV Coherence (когерентность CV)** - мера согласованности тем на основе совместной встречаемости слов
4. **Topic Size Distribution (распределение размеров тем)** - равномерность распределения документов по темам
5. **Silhouette Score (коэффициент силуэта)** - оценка качества кластеризации
6. **Outlier Percentage (процент выбросов)** - доля документов, не относящихся к конкретной теме
7. **Intertopic Distance (межтематическое расстояние)** - среднее расстояние между центрами тем












In [60]:

    
print("📊 Выполнение комплексной оценки...")

# Создаем оценщик
evaluator = TopicModelEvaluator(
    topic_model=topic_model,
    texts=to_proccess_text,
    topics=topics,
    embeddings=embeddings
)

# Выполняем комплексную оценку
results = evaluator.evaluate_comprehensive()

# Генерируем отчет
report = evaluator.generate_evaluation_report(
save_path="../data/interim/topic_evaluation_report.csv"
)

print("\n📈 Результаты оценки:")
print("=" * 50)

for metric, info in report.iterrows():
    print(f"{metric:30s}: {info['Value']:10.4f} - {info['Interpretation']}")

# Дополнительная оценка CV Coherence
coherence_results = evaluator.calculate_cv_coherence()
print(f"\n🎯 Средняя CV Coherence тем: {coherence_results.get('avg_cv_coherence', 0):.4f}")


📊 Выполнение комплексной оценки...


Ошибка при вычислении когерентности для темы 2: unable to interpret topic as either a list of tokens or a list of ids
Ошибка при вычислении когерентности для темы 3: unable to interpret topic as either a list of tokens or a list of ids
Ошибка при вычислении когерентности для темы 4: unable to interpret topic as either a list of tokens or a list of ids
Ошибка при вычислении когерентности для темы 7: unable to interpret topic as either a list of tokens or a list of ids
Ошибка при вычислении когерентности для темы 9: unable to interpret topic as either a list of tokens or a list of ids
Ошибка при вычислении когерентности для темы 12: unable to interpret topic as either a list of tokens or a list of ids
Ошибка при вычислении когерентности для темы 13: unable to interpret topic as either a list of tokens or a list of ids
Ошибка при вычислении когерентности для темы 14: unable to interpret topic as either a list of tokens or a list of ids
Ошибка при вычислении когерентности для темы 17: unab


📈 Результаты оценки:
n_topics                      :    28.0000 - Количество обнаруженных тем
outlier_ratio                 :     0.1753 - Доля документов-выбросов (чем меньше, тем лучше)
avg_topic_size                :    26.2143 - Специфическая метрика
std_topic_size                :    30.7099 - Специфическая метрика
silhouette_score              :     0.0588 - Качество кластеризации (-1 до 1, выше лучше)
calinski_harabasz_score       :    13.7684 - Отношение межгрупповой/внутригрупповой дисперсии (выше лучше)
davies_bouldin_score          :     2.4647 - Средняя схожесть кластеров (ниже лучше)
avg_topic_representation_length:    10.0000 - Специфическая метрика
std_topic_representation_length:     0.0000 - Специфическая метрика
topic_word_uniqueness         :     0.9571 - Уникальность слов в темах (выше лучше)
topic_entropy                 :     4.1434 - Энтропия распределения тем (выше = более равномерно)
topic_gini_coefficient        :     0.4852 - Неравномерность распределения (0

Ошибка при вычислении когерентности для темы 2: unable to interpret topic as either a list of tokens or a list of ids
Ошибка при вычислении когерентности для темы 3: unable to interpret topic as either a list of tokens or a list of ids
Ошибка при вычислении когерентности для темы 4: unable to interpret topic as either a list of tokens or a list of ids
Ошибка при вычислении когерентности для темы 7: unable to interpret topic as either a list of tokens or a list of ids


KeyboardInterrupt: 

## Отрисовка кластеров

In [68]:
# Визуализация кластеров новостей в 2D пространстве с помощью plotly
import plotly.express as px
import pandas as pd
import numpy as np

# Получаем координаты документов в 2D пространстве
embeddings_2d = zipped_embs

# Получаем данные о документах
doc_info = topic_model.get_document_info(data['cleaned_text'].tolist())

# Создаем DataFrame для визуализации
plot_df = pd.DataFrame({
    'x': embeddings_2d.embedding_x,
    'y': embeddings_2d.embedding_y,
    'topic': embeddings_2d.topic,
    'text': doc_info.Document,
    'topic_name': doc_info.Name
})

# Создаем цветовую схему
colors = px.colors.qualitative.Plotly

# Создаем интерактивную визуализацию
fig = px.scatter(
    plot_df, 
    x='x', 
    y='y', 
    color='topic_name',
    hover_data=['text'],
    title='Кластеризация новостей по темам',
    color_discrete_sequence=colors,
    opacity=0.7,
    size_max=10
)

# Настраиваем внешний вид графика
fig.update_traces(marker=dict(size=8, line=dict(width=1, color='DarkSlateGrey')))
fig.update_layout(
    legend_title_text='Темы',
    xaxis_title="",
    yaxis_title="",
    xaxis=dict(showticklabels=False),
    yaxis=dict(showticklabels=False),
    plot_bgcolor='white'
)

# Отображаем график
fig.show()



AttributeError: 'numpy.ndarray' object has no attribute 'embedding_x'